# 02 - Onboard and Score an H2O Binary Model

This self-contained notebook starts from an existing H2O binary model bundle. It validates the model and checksums, enforces the training H2O version, loads the model with `h2o.load_model()`, compares golden predictions, and can optionally register the folder as an Azure ML custom model.

It does **not** train or export a model.

## Configure the Artifact

The defaults consume Notebook 01 output. Copy `.env.example` to `.env` at the repository root and set the shared Azure ML values there. Existing process environment variables take precedence.

Keep `REGISTER_IN_AZURE=false` for local validation. Set it to `true` only when the validated model bundle should be registered in the configured workspace.

H2O binary models are version-specific. The notebook's H2O package must exactly match the version recorded in the model manifest.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys

import numpy as np
import pandas as pd
from dotenv import load_dotenv

for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (folder / "data" / "taxi-data" / "raw" / "yellowTaxiData.csv").is_file():
        REPO_ROOT = folder
        break
else:
    raise FileNotFoundError("Run this notebook from inside the MLOPs-AzureML repository")

ENV_FILE = REPO_ROOT / ".env"
load_dotenv(ENV_FILE)

def env_flag(name, default=False):
    value = os.getenv(name)
    if value is None:
        return default

    normalized = value.strip().lower()
    if normalized not in {"1", "0", "true", "false", "yes", "no", "on", "off"}:
        raise ValueError(f"{name} must be true or false")
    return normalized in {"1", "true", "yes", "on"}


# The local JDK is used by H2O internally; notebook code stays in the Python API.
if not shutil.which("java"):
    java_bins = sorted((Path.home() / ".jdk").glob("jdk-17*/bin"), reverse=True)
    if java_bins:
        os.environ["JAVA_HOME"] = str(java_bins[0].parent)
        os.environ["PATH"] = str(java_bins[0]) + os.pathsep + os.environ["PATH"]

CONFIG = {
    "bundle_dir": REPO_ROOT / "tmp" / "h2o_binary" / "taxi_fare",
    "model_path": None,  # None uses model_file from the manifest.
    "input_csv": REPO_ROOT / "tmp" / "h2o_binary" / "taxi_fare" / "golden_input.csv",
    "expected_csv": REPO_ROOT / "tmp" / "h2o_binary" / "taxi_fare" / "golden_expected.csv",
    "output_csv": REPO_ROOT / "tmp" / "h2o_binary" / "onboarding_predictions.csv",
    "manifest_path": REPO_ROOT / "tmp" / "h2o_binary" / "taxi_fare" / "model_manifest.json",
    "model_name": "taxi-fare-h2o-binary",
    "model_version": "1",
    "register_in_azure": env_flag("REGISTER_IN_AZURE"),
    "subscription_id": os.getenv("AZURE_SUBSCRIPTION_ID", "").strip(),
    "tenant_id": os.getenv("AZURE_TENANT_ID", "").strip(),
    "resource_group": os.getenv("AZURE_RESOURCE_GROUP", "").strip(),
    "workspace_name": os.getenv("AZUREML_WORKSPACE_NAME", "").strip(),
}

if CONFIG["register_in_azure"]:
    required_settings = {
        "AZURE_SUBSCRIPTION_ID": CONFIG["subscription_id"],
        "AZURE_RESOURCE_GROUP": CONFIG["resource_group"],
        "AZUREML_WORKSPACE_NAME": CONFIG["workspace_name"],
    }
    missing_settings = [name for name, value in required_settings.items() if not value]
    if missing_settings:
        raise ValueError(
            f"Set {', '.join(missing_settings)} in {ENV_FILE.name} or the process environment"
        )

print(f"Python: {sys.version.split()[0]}")
{
    "bundle_dir": str(CONFIG["bundle_dir"].relative_to(REPO_ROOT)),
    "model_name": CONFIG["model_name"],
    "model_version": CONFIG["model_version"],
    "register_in_azure": CONFIG["register_in_azure"],
}

## Validate the Binary Model Bundle

Preflight verifies the model file, input fixture, expected predictions, checksums, immutable asset version, binary format marker, and exact H2O compatibility version before loading anything.

In [ ]:
bundle_dir = Path(CONFIG["bundle_dir"]).resolve()
manifest_path = Path(CONFIG["manifest_path"]).resolve()
input_path = Path(CONFIG["input_csv"]).resolve()
expected_path = Path(CONFIG["expected_csv"]).resolve()
output_path = Path(CONFIG["output_csv"]).resolve()

if not manifest_path.is_file():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
model_path = (
    Path(CONFIG["model_path"]).resolve()
    if CONFIG["model_path"]
    else bundle_dir / manifest["model_file"]
)

required = {
    "binary model": model_path,
    "input CSV": input_path,
    "expected predictions": expected_path,
}
missing = [name for name, path in required.items() if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing bundle files: " + ", ".join(missing))
if CONFIG["model_version"].lower() == "latest":
    raise ValueError("Use an immutable model version, not 'latest'")
if manifest.get("model_format") != "h2o_binary":
    raise ValueError("The manifest does not describe an H2O binary model")

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for path in (model_path, input_path, expected_path):
    expected_hash = manifest.get("files", {}).get(path.name)
    if not expected_hash:
        raise ValueError(f"Manifest has no checksum for {path.name}")
    if sha256(path) != expected_hash:
        raise ValueError(f"Checksum mismatch: {path.name}")

input_frame = pd.read_csv(input_path)
features = manifest["features"]
if list(input_frame.columns) != features:
    raise ValueError(f"Expected columns in this order: {features}")

print({
    "model": model_path.name,
    "format": manifest["model_format"],
    "h2o_version": manifest["h2o_version"],
    "rows": len(input_frame),
    "features": features,
    "checksums": "passed",
})

## Load and Score the Binary Model

The notebook starts H2O, loads the binary model with `h2o.load_model()`, and checks its predictions against the saved golden values.

In [ ]:
import h2o

required_h2o_version = manifest["h2o_version"]
if h2o.__version__ != required_h2o_version:
    raise RuntimeError(
        f"This model requires h2o=={required_h2o_version}; found {h2o.__version__}"
    )

h2o.init(max_mem_size="2G", nthreads=-1)
loaded_model = h2o.load_model(str(model_path))

scoring_frame = h2o.H2OFrame(input_frame)
for column in manifest.get("categorical_features", []):
    scoring_frame[column] = scoring_frame[column].asfactor()

expected = pd.read_csv(expected_path)
actual = loaded_model.predict(scoring_frame).as_data_frame()
actual.to_csv(output_path, index=False)

if len(expected) != len(actual):
    raise ValueError(f"Expected {len(expected)} predictions, received {len(actual)}")
np.testing.assert_allclose(expected["predict"], actual["predict"], rtol=1e-6, atol=1e-6)

comparison = pd.DataFrame(
    {
        "expected": expected["predict"],
        "actual": actual["predict"],
        "absolute_error": (expected["predict"] - actual["predict"]).abs(),
    }
)
display(comparison)
print(f"Maximum absolute error: {comparison['absolute_error'].max():.8f}")

## Optional: Register in Azure ML

Registration uploads the validated binary-model bundle as a `custom_model`. Its Azure ML scoring environment must pin the exact H2O version recorded in the manifest.

In [ ]:
if CONFIG["register_in_azure"]:
    from azure.ai.ml import MLClient
    from azure.ai.ml.constants import AssetTypes
    from azure.ai.ml.entities import Model
    from azure.identity import AzureCliCredential

    credential = AzureCliCredential(tenant_id=CONFIG["tenant_id"] or None)
    ml_client = MLClient(
        credential,
        CONFIG["subscription_id"],
        CONFIG["resource_group"],
        CONFIG["workspace_name"],
    )
    registered_model = ml_client.models.create_or_update(
        Model(
            path=str(bundle_dir),
            name=CONFIG["model_name"],
            version=CONFIG["model_version"],
            type=AssetTypes.CUSTOM_MODEL,
            description="Validated H2O binary model bundle",
            tags={
                "model_format": manifest["model_format"],
                "h2o_version": manifest["h2o_version"],
                "model_sha256": sha256(model_path),
                "golden_test": "passed",
            },
        )
    )
    print(f"Registered model: {registered_model.name}:{registered_model.version}")
else:
    print("Azure registration skipped. Set REGISTER_IN_AZURE=true in .env when ready.")

## Finish

The validated binary model is ready for an Azure ML environment pinned to the recorded H2O version. Production promotion must reuse the exact model file and checksum.

In [ ]:
h2o.cluster().shutdown(prompt=False)